# محور التقييم — المرحلة ٠ من مشروع اكتشاف الإشارة

دفتر مستقلّ تماماً عن `crypto_data_pipeline` — لا يُنتج ميزة ولا نموذجاً، بل **يجيب سؤالاً واحداً لكل فرضية تُعرَض عليه:** هل هذا أفضل من الصدفة، بدليل رقمي لا انطباع؟

**القاعدة الحاكمة (من خطة المشروع):** لا فرضية تُقبل أو تُرفض إلا عبر هذا المحور. لا استثناء، ولا "تبدو واعدة".

## البنية

1. **`core.py` (منطقياً — هنا كخلايا)** — ثلاث دوال مستقلّة تماماً عن شكل بيانات دفتر التحضير: `compute_ic`، `decile_spread`، `permutation_baseline`. تُختبَر ببيانات تركيبية محضة، بلا أي اعتماد على `rolling_splits` أو بنية `dataset`.
2. **`windows.py`** — `evaluate_windows`: يُجمِّع نتائج (١) عبر عدّة نوافذ زمنية، ويحكم على **الاتساق** لا رقماً واحداً.
3. **`integration.py`** — الطبقة الوحيدة التي تفترض شكل مخرجات `rolling_splits`/`build_dataset`. أي خطأ في هذا الافتراض محصور هنا، لا يمسّ صحّة القياس نفسه في (١) و(٢).

## بوّابة الخروج (قبل الوثوق بأي نتيجة لاحقة)

شغّل `momentum_predict_fn` (فرضية معروفة الاتجاه العام أكاديمياً: استمرار آخر حركة) عبر `evaluate_hypothesis_over_rolling_windows` على بياناتك الحقيقية. IC صغير موجب متّسق عبر أغلب النوافذ = المحور سليم. لا شيء أو عشوائي = أصلح القياس قبل اختبار أي فرضية أخرى — راجع القسم الأخير "بوّابة المرحلة ٠" لتشغيلها فعلياً.

## ١) استيرادات


In [ ]:
from __future__ import annotations
from typing import Optional, List, Tuple, Dict, Any, Callable

import numpy as np
import pandas as pd


## ٢) معامل الارتباط (Information Coefficient)

الأداة الأساسية لكل شيء لاحق: رقم واحد يلخّص "هل ترتيب التنبؤات يطابق ترتيب العوائد الفعلية؟" — لا أكثر ولا أقل.

In [ ]:
def compute_ic(predictions: np.ndarray, actuals: np.ndarray,
               method: str = "spearman", min_samples: int = 10) -> float:
    """معامل الارتباط بين تنبؤات فرضية والعوائد الفعلية المحقَّقة.

    ``method='spearman'`` (افتراضي): ارتباط الرتب — لا يفترض علاقة خطية بين
    التنبؤ والعائد، فقط أن الترتيب صحيح (تنبؤ أعلى ↔ عائد أعلى غالباً). هذا
    أنسب لتقييم فرضية قد تُنتج "درجة" غير مُعايَرة (لا سعراً حقيقياً)،
    ولمقاومته الأفضل للقيم الشاذة القليلة الشائعة في عوائد الأصول المالية.
    ``method='pearson'``: ارتباط خطي كلاسيكي — أضيق افتراضاً (يفترض علاقة
    خطية فعلاً)، لكن أسرع حساباً وأكثر حساسية لعلاقة خطية قوية فعلية.

    يتجاهل صمتاً أي عيّنة NaN في أي من المصفوفتين (لا يرفضها العملية كلها،
    ولا يُصفّرها بصمت أيضاً — ببساطة تُستبعَد من حساب الارتباط). يرفع
    ``ValueError`` إن قلّ عدد العيّنات الصالحة عن ``min_samples`` — IC على
    عيّنات قليلة جداً رقم عشوائي بلا معنى إحصائي، لا نتيجة تستحق الثقة.

    Returns:
        رقم واحد بين -1 و1. صفر بالضبط (لا NaN) إن كانت إحدى المصفوفتين
        ثابتة تماماً (تباين صفري) بعد إزالة NaN — لا ارتباط مُعرَّف رياضياً
        مع قيمة ثابتة، وصفر هنا هو الإعلان الصريح "لا إشارة" لا خطأ حسابي.
    """
    predictions = np.asarray(predictions, dtype="float64")
    actuals = np.asarray(actuals, dtype="float64")
    if predictions.shape != actuals.shape:
        raise ValueError(f"الشكلان يجب أن يتطابقا: {predictions.shape} != {actuals.shape}")
    if method not in ("spearman", "pearson"):
        raise ValueError(f"method غير معروفة: {method!r} — المتاح: spearman, pearson")

    mask = np.isfinite(predictions) & np.isfinite(actuals)
    n = int(mask.sum())
    if n < min_samples:
        raise ValueError(
            f"❌ {n} عيّنة صالحة فقط (بعد استبعاد NaN) — أقل من الحدّ الأدنى "
            f"{min_samples}. IC على عدد بهذا الصغر رقم عشوائي لا يُعتمَد عليه.")

    p, a = predictions[mask], actuals[mask]
    if np.std(p) < 1e-12 or np.std(a) < 1e-12:
        return 0.0

    if method == "spearman":
        p = pd.Series(p).rank().to_numpy()
        a = pd.Series(a).rank().to_numpy()
    corr = np.corrcoef(p, a)[0, 1]
    return float(corr) if np.isfinite(corr) else 0.0


# ══════════════════════════════════════════════════════════════════════════
# ٢) فحص العُشر (decile spread)
# ══════════════════════════════════════════════════════════════════════════



## ٣) فحص العُشر (Decile Spread)

يكشف إشارة قد لا يظهر أثرها في IC واحد يلخّص كل العيّنات — التركّز في الأطراف.

In [ ]:
def decile_spread(predictions: np.ndarray, actuals: np.ndarray,
                  n_deciles: int = 10, min_per_decile: int = 5) -> Dict[str, Any]:
    """يقسّم العيّنات إلى ``n_deciles`` حسب رتبة التنبؤ، ويحسب متوسط العائد
    الفعلي لكل عُشر — يكشف إشارة حقيقية حتى لو كان IC الكلي صغيراً (تركيز
    الأثر في الأطراف لا يظهر بالضرورة في معامل ارتباط واحد يلخّص كل العيّنات).

    ``monotonic``: هل متوسط العائد يزداد (أو يتناقص) بانتظام من أدنى عُشر
    لأعلاه؟ **هذا أهم من ``spread`` وحده**: فرق كبير بين الطرفين مع منتصف
    عشوائي غير مرتّب مرشّح أقوى لأن يكون صدفة من فرق نظيف رتيب عبر كل الأعشار.

    Args:
        min_per_decile: أقل عدد عيّنات مقبول في كل عُشر — دون ذلك يُرفَع
            ``ValueError`` بدل نتيجة على عيّنات قليلة جداً لا تمثّل شيئاً.

    Returns:
        قاموس ``{'deciles': DataFrame(index, mean_actual, n), 'spread':
        float, 'monotonic': bool, 'top_minus_bottom_t_stat': float}``.
        ``top_minus_bottom_t_stat`` تقريب سريع (فرق المتوسطات ÷ الخطأ
        المعياري المجمَّع) — ليس اختباراً إحصائياً صارماً، فقط مؤشر أوّلي
        لحجم الفرق نسبة لتشتّت كل عُشر؛ الحسم الفعلي عبر ``permutation_baseline``.
    """
    predictions = np.asarray(predictions, dtype="float64")
    actuals = np.asarray(actuals, dtype="float64")
    mask = np.isfinite(predictions) & np.isfinite(actuals)
    p, a = predictions[mask], actuals[mask]
    n = len(p)
    if n < n_deciles * min_per_decile:
        raise ValueError(
            f"❌ {n} عيّنة صالحة لا تكفي لـ{n_deciles} أعشار بحدّ أدنى "
            f"{min_per_decile} لكل عُشر (تحتاج {n_deciles * min_per_decile} على الأقل).")

    order = np.argsort(p, kind="stable")
    bucket = np.empty(n, dtype="int64")
    bucket[order] = (np.arange(n) * n_deciles) // n

    rows = []
    for d in range(n_deciles):
        vals = a[bucket == d]
        rows.append({"decile": d, "mean_actual": float(vals.mean()),
                    "std_actual": float(vals.std()), "n": int(len(vals))})
    table = pd.DataFrame(rows)

    means = table["mean_actual"].to_numpy()
    diffs = np.diff(means)
    monotonic = bool(np.all(diffs >= 0) or np.all(diffs <= 0))
    spread = float(means[-1] - means[0])

    top, bottom = a[bucket == n_deciles - 1], a[bucket == 0]
    pooled_se = np.sqrt(top.var(ddof=1) / len(top) + bottom.var(ddof=1) / len(bottom))
    t_stat = float(spread / pooled_se) if pooled_se > 1e-12 else 0.0

    return {"deciles": table, "spread": spread, "monotonic": monotonic,
           "top_minus_bottom_t_stat": t_stat}


# ══════════════════════════════════════════════════════════════════════════
# ٣) خطّ أساس عشوائي (permutation baseline)
# ══════════════════════════════════════════════════════════════════════════



## ٤) خطّ أساس عشوائي (Permutation Baseline)

الإجابة المباشرة على "أفضل من الصدفة؟" — لا الاكتفاء برقم IC مجرَّد.

In [ ]:
def permutation_baseline(predictions: np.ndarray, actuals: np.ndarray,
                         n_shuffles: int = 1000, method: str = "spearman",
                         seed: Optional[int] = None) -> Dict[str, Any]:
    """يبني توزيعاً خالياً (null distribution) لـIC عبر بعثرة ``actuals``
    عشوائياً ``n_shuffles`` مرة (بلا تغيير ``predictions``)، ويقارن IC
    الحقيقي بهذا التوزيع — الإجابة المباشرة على "هل هذا أفضل من الصدفة؟"
    بدل الاكتفاء برقم IC مجرَّد.

    ``percentile``: أين يقع IC الحقيقي ضمن توزيع الصدفة (0-100). قريب من
    100 (أو 0 لإشارة سالبة قوية) = IC الحقيقي أقوى من الغالبية الساحقة من
    عمليات البعثرة العشوائية. ``p_value``: تقريب أحادي الجانب — نسبة عمليات
    البعثرة التي أعطت |IC| ≥ |IC الحقيقي| (اختبار ثنائي الطرف ضمنياً، يلائم
    عدم معرفة اتجاه الإشارة مسبقاً في هذا السياق الاستكشافي).

    ``seed``: لتكرار نفس نتيجة البعثرة بالضبط — مهم لسجلّ التجارب (نتيجة
    فرضية يجب أن تكون قابلة لإعادة الإنتاج، لا تتغيّر بين تشغيلين).
    """
    predictions = np.asarray(predictions, dtype="float64")
    actuals = np.asarray(actuals, dtype="float64")
    real_ic = compute_ic(predictions, actuals, method=method)

    mask = np.isfinite(predictions) & np.isfinite(actuals)
    p, a = predictions[mask], actuals[mask]
    rng = np.random.default_rng(seed)

    null_ics = np.empty(n_shuffles, dtype="float64")
    for i in range(n_shuffles):
        shuffled = rng.permutation(a)
        null_ics[i] = compute_ic(p, shuffled, method=method, min_samples=1)

    percentile = float((null_ics < real_ic).mean() * 100.0)
    p_value = float((np.abs(null_ics) >= abs(real_ic)).mean())

    return {"real_ic": real_ic, "null_mean": float(null_ics.mean()),
           "null_std": float(null_ics.std()), "percentile": percentile,
           "p_value": p_value, "n_shuffles": n_shuffles}


## ٥) تجميع التقييم عبر عدّة نوافذ زمنية

الاتساق عبر نوافذ متعدّدة (عبر `rolling_splits`) دليل أقوى بكثير من IC قويّ على تقسيم واحد قد يكون حظّاً.

In [ ]:
# -*- coding: utf-8 -*-
"""تجميع التقييم عبر عدّة نوافذ زمنية — يستهلك مخرجات core.py لكل نافذة على
حدة، ثم يُلخّص الاتساق عبرها. الاتساق عبر عدّة نوافذ دليل أقوى بكثير من IC
قوي على تقسيم واحد قد يكون حظّاً (راجع منهجية التقييم في خطة المشروع)."""
from __future__ import annotations
from typing import Optional, List, Tuple, Dict, Any
import numpy as np
import pandas as pd



def evaluate_windows(window_results: List[Tuple[str, np.ndarray, np.ndarray]],
                     ic_method: str = "spearman", n_shuffles: int = 1000,
                     min_samples: int = 10, seed: Optional[int] = None,
                     verbose: bool = True) -> Dict[str, Any]:
    """يقيس فرضية واحدة عبر عدّة نوافذ (كل عنصر: اسم النافذة، تنبؤات، عوائد
    فعليّة) — لكل نافذة IC + عُشر + خطّ أساس عشوائي على حدة، ثم تلخيص شامل.

    نوافذ فشلت (بيانات غير كافية) تُستبعَد من التلخيص مع تحذير صريح، لا
    تُسقِط العملية كلها — فرضية قد تنجح على أغلب النوافذ حتى لو فشلت واحدة
    بسبب نقص بيانات محلي (عملة جديدة الإدراج في تلك الفترة مثلاً).

    Returns:
        قاموس يحوي ``per_window`` (تفصيل كل نافذة)، و``mean_ic``/``std_ic``
        (عبر النوافذ الناجحة)، و``frac_significant`` (نسبة النوافذ التي
        تجاوزت الصدفة بـp<0.05 وبنفس اتجاه IC الكلي — لا يكفي p صغيراً وحده،
        الاتجاه المتّسق هو ما يهمّ)، و``consistent_sign`` (هل كل النوافذ
        الناجحة اتّفقت على إشارة IC، موجبة كانت أم سالبة؟).
    """
    per_window = []
    for name, preds, actuals in window_results:
        try:
            ic = compute_ic(preds, actuals, method=ic_method, min_samples=min_samples)
            dec = decile_spread(preds, actuals, min_per_decile=max(1, min_samples // 10) or 1)
            perm = permutation_baseline(preds, actuals, n_shuffles=n_shuffles,
                                        method=ic_method, seed=seed)
            per_window.append({"window": name, "status": "ok", "ic": ic,
                               "spread": dec["spread"], "monotonic": dec["monotonic"],
                               "p_value": perm["p_value"], "percentile": perm["percentile"]})
        except ValueError as exc:
            per_window.append({"window": name, "status": "skipped", "note": str(exc)[:120]})
            if verbose:
                print(f"   ⚠️ [{name}] تُخطّيت: {str(exc)[:120]}")

    ok = [w for w in per_window if w["status"] == "ok"]
    table = pd.DataFrame(per_window)

    if not ok:
        if verbose:
            print("❌ لا نافذة واحدة ناجحة — لا يمكن تلخيص شيء.")
        return {"per_window": table, "mean_ic": None, "std_ic": None,
               "frac_significant": None, "consistent_sign": None, "n_ok": 0}

    ics = np.array([w["ic"] for w in ok])
    mean_ic, std_ic = float(ics.mean()), float(ics.std())
    overall_sign = np.sign(mean_ic)
    consistent_sign = bool(np.all(np.sign(ics) == overall_sign)) if overall_sign != 0 else False
    frac_significant = float(np.mean([
        w["p_value"] < 0.05 and np.sign(w["ic"]) == overall_sign for w in ok]))

    if verbose:
        print(f"📊 {len(ok)}/{len(window_results)} نافذة ناجحة | "
              f"IC: متوسط={mean_ic:.4f} انحراف={std_ic:.4f} | "
              f"اتّساق الإشارة عبر كل النوافذ: {'نعم' if consistent_sign else 'لا'} | "
              f"نوافذ معنوية (p<0.05) بنفس الاتجاه: {frac_significant:.0%}")

    return {"per_window": table, "mean_ic": mean_ic, "std_ic": std_ic,
           "frac_significant": frac_significant, "consistent_sign": consistent_sign,
           "n_ok": len(ok)}


## ٦) طبقة التكامل مع دفتر التحضير

**العقد المفترض — تحقّقتُه من مصدر `_take`/`add_y_prefix` الفعلي في دفتر التحضير مباشرة، بعد أن أخطأتُ فيه مرّتين بالتخمين:** كل قسم قاموس فيه `X_{tf}` (مصفوفة `(N,T,F)`)، و`base_params`/`last_candles` (علامة "قسم مسطّح")، و**الأهداف متداخلة تحت مفتاح فرعي واحد `y`** — `split['y']['y_close_reg']`، لا `split['y_close_reg']` مباشرة. لا `feature_order` على مستوى القسم إطلاقاً (فقط في `dataset` الأصلي قبل التقطيع — مرّرها صراحةً). `test` قد يكون قاموساً مسطّحاً كما فوق، أو `{اسم_الأصل: قسم}` (`keep_asset_test_separate`).

⚠️ هذا القسم وحده يفترض شكل بيانات دفتر التحضير — لو تغيّر شكل مخرجات `rolling_splits` مستقبلاً، هنا فقط يحتاج تعديلاً، لا الأقسام (٢)-(٥).

In [ ]:
# -*- coding: utf-8 -*-
"""طبقة التكامل مع دفتر التحضير (crypto_data_pipeline) — الجزء الوحيد الذي
يفترض شكل بيانات معيّناً (مخرجات rolling_splits/_take). بمعزل عمداً عن
core.py وwindows.py (المُختبَرين بلا أي اعتماد على هذا الشكل) — أي خلل في
افتراض هنا محصور في هذا الملف، لا يمسّ صحّة القياس نفسه.

**العقد المفترض — تحقّقتُه من مصدر ``_take``/``add_y_prefix`` الفعلي مباشرة،
لا تخميناً:** كل قسم (من ``_take`` داخل ``rolling_splits``/``split_data``) قاموس:

* ``base_params``, ``last_candles``: مصفوفات (N, ...) — موجودتان دائماً، تُستخدَمان
  هنا كعلامة "هذا قسم بيانات مسطّح" (خلاف قاموس أصول ``{اسم: قسم}``).
* ``X_{tf}``: مصفوفة (N,T,F) لكل فريم — مسطّحة كما هو متوقَّع.
* ``y``: **قاموس فرعي واحد** ``{اسم_الهدف: مصفوفة (N,)}`` — لا مفاتيح
  ``y_head`` مباشرة على مستوى القسم نفسه؛ كلها متداخلة تحت هذا المفتاح
  الواحد (تصميم متعمَّد في الدفتر الأصلي، توافقاً مع طريقة Keras في مطابقة
  مخرجات متعدّدة بالاسم). ``add_y_prefix`` (إن ``prefix_y=True``، الافتراضي)
  يُضيف بادئة ``y_`` لمفاتيح *داخل* هذا القاموس الفرعي فقط.
* لا ``feature_order`` على مستوى القسم — موجودة فقط في ``dataset`` الأصلي
  قبل التقطيع (مؤكَّد سابقاً من تشغيل حقيقي).

``test`` قد يكون قسماً مسطّحاً كما فوق، أو ``{اسم_الأصل: قسم}``
(``keep_asset_test_separate=True``) — كل قيمة فيه قسم مسطّح بنفس الشكل.
"""
from __future__ import annotations
from typing import Optional, List, Tuple, Dict, Any, Callable
import numpy as np



def concat_splits(split: Any) -> Dict[str, Any]:
    """يُسطّح ``test`` إن كان ``{اسم_الأصل: قسم}`` (keep_asset_test_separate)
    بدمج كل الأصول في قاموس واحد — X مُلحَقة بمحور العيّنات، وy (القاموس
    الفرعي) تُدمَج مفتاحاً مفتاحاً. قسم مسطّح أصلاً يُعاد كما هو بلا نسخ.

    ✅ **التمييز بمفتاح ``'base_params'`` تحديداً** — موجود في كل قسم مسطّح
    (يضعه ``_take`` دائماً)، وغير موجود أبداً بين أسماء الأصول في قاموس
    ``{اسم: قسم}``. أدقّ من التخمين بشكل القيمة (``isinstance(..., dict)``
    وحده لا يكفي: قيمة ``split['y']`` في القسم المسطّح نفسه قاموس أيضاً،
    فقد يُلتبَس بقاموس أصول لو اعتُمد نوع القيمة فقط — راجع سجلّ التعديلات).
    """
    if not isinstance(split, dict) or not split:
        raise ValueError(f"split فارغ أو ليس قاموساً (النوع: {type(split).__name__}).")

    if "base_params" in split:
        return split          # مسطّح أصلاً

    parts = list(split.items())
    bad = next((name for name, p in parts
               if not (isinstance(p, dict) and "base_params" in p)), None)
    if bad is not None:
        raise ValueError(
            f"العنصر '{bad}' لا يبدو قسم بيانات صالحاً (بلا 'base_params') — "
            f"هل split فعلاً {{اسم_الأصل: قسم}} كما هو مفترض؟")

    split_parts = [p for _, p in parts]
    out: Dict[str, Any] = {
        "base_params": np.concatenate([p["base_params"] for p in split_parts], axis=0),
        "last_candles": np.concatenate([p["last_candles"] for p in split_parts], axis=0),
    }
    for k in [k for k in split_parts[0] if k.startswith("X_")]:
        out[k] = np.concatenate([p[k] for p in split_parts if k in p], axis=0)

    y_keys = set()
    for p in split_parts:
        y_keys.update((p.get("y") or {}).keys())
    out["y"] = {yk: np.concatenate([p["y"][yk] for p in split_parts if yk in (p.get("y") or {})],
                                  axis=0)
               for yk in y_keys}
    return out


def extract_actuals(split: Dict[str, Any], target_key: str = "y_close_reg") -> np.ndarray:
    """يقرأ مصفوفة الهدف الفعلي (العائد المحقَّق) من ``split['y'][...]`` —
    ✅ **الأهداف متداخلة تحت مفتاح فرعي واحد ``'y'``**، لا مفاتيح ``y_head``
    مباشرة على القسم نفسه (تأكّدتُ من هذا من مصدر ``_take``/``add_y_prefix``
    الفعلي — راجع توثيق الوحدة أعلى الملف). يقبل ``target_key`` مع البادئة
    ``'y_'`` أو بدونها؛ يجرّب كليهما (``add_y_prefix`` قد يكون فعل أو لم يفعل
    حسب ``prefix_y`` عند بناء النافذة).
    """
    split = concat_splits(split)
    y = split.get("y")
    if not isinstance(y, dict):
        raise ValueError(
            "لا مفتاح 'y' (قاموس أهداف فرعي) في هذا القسم — تحقّق أنك مرّرت "
            "قسماً من مخرجات rolling_splits/split_data مباشرة، لا شيئاً آخر.")
    alt_key = target_key[len("y_"):] if target_key.startswith("y_") else f"y_{target_key}"
    for k in (target_key, alt_key):
        if k in y:
            return np.asarray(y[k], dtype="float64")
    raise KeyError(f"لا '{target_key}' ولا '{alt_key}' في split['y'] — المتاح: {list(y.keys())}")


def extract_feature_last_diff(split: Dict[str, Any], feature: str = "close",
                              tf: Optional[str] = None,
                              feature_order: Optional[List[str]] = None) -> np.ndarray:
    """فرق آخر خطوتين زمنيتين لميزة واحدة داخل نافذة كل عيّنة — "أحدث تغيّر"
    لتلك الميزة، مقياس خام لكن كافٍ تماماً لأي فرضية تُقيَّم بـIC سبيرمان
    (ارتباط رتبي لا يتأثّر بالمقياس المطلق، فلا حاجة لعكس التطبيع إلى سعر
    حقيقي — أي تحويل رتيب لنفس الترتيب يعطي نفس IC).

    ✅ **``feature_order`` غير محفوظة داخل نوافذ ``rolling_splits``** (قِيس
    فعلياً: موجودة في ``dataset['feature_order']`` الأصلي، لا في كل قسم
    ``train``/``val``/``test`` الناتج عن التقطيع) — مرّرها صراحةً من
    ``dataset`` الأصلي. القسم نفسه يبقى يُحاوَل أولاً (توافقاً مع أي مصدر
    بيانات يحفظها فعلاً)، لكن التمرير الصريح هو المسار الموثوق.

    ``tf``: الفريم المطلوب (افتراضياً أول عمود X_* موجود في القسم).
    ``feature``: اسمها كما في ``feature_order`` بالضبط.
    """
    split = concat_splits(split)
    feature_order = feature_order if feature_order is not None else split.get("feature_order")
    if not feature_order:
        raise ValueError(
            "لا 'feature_order' — لا في القسم ولا مُمرَّرة صراحةً. مرّرها من "
            "dataset الأصلي: extract_feature_last_diff(split, feature_order=dataset['feature_order']).")
    if feature not in feature_order:
        raise KeyError(f"'{feature}' غير موجودة في feature_order ({len(feature_order)} ميزة).")
    idx = feature_order.index(feature)

    if tf is None:
        x_keys = [k for k in split if k.startswith("X_")]
        if not x_keys:
            raise ValueError("لا عمود X_* في قسم البيانات.")
        tf = x_keys[0][2:]
    X = np.asarray(split[f"X_{tf}"])
    if X.shape[-1] <= idx:
        raise ValueError(f"عمود X_{tf} له {X.shape[-1]} ميزة فقط، لا يكفي للفهرس {idx}.")
    return X[:, -1, idx] - X[:, -2, idx]


def momentum_predict_fn(train: Dict, val: Dict, test: Dict,
                        feature: str = "close", tf: Optional[str] = None,
                        feature_order: Optional[List[str]] = None) -> np.ndarray:
    """فرضية الأساس (المرحلة ٠، اختبار سلامة المحور): التنبؤ = آخر تغيّر في
    السعر داخل نافذة كل عيّنة (استمرار الاتجاه الأخير) — لا تدريب، لا تحتاج
    ``train``/``val`` إطلاقاً، تُنتَج مباشرة من ``test``.

    مرّر ``feature_order`` (من ``dataset['feature_order']`` الأصلي) عبر
    ``functools.partial`` قبل تمريرها لـ``evaluate_hypothesis_over_rolling_windows``
    — راجع مثال الاستخدام في نهاية الدفتر.
    """
    return extract_feature_last_diff(test, feature=feature, tf=tf, feature_order=feature_order)


def evaluate_hypothesis_over_rolling_windows(
    windows: List[Tuple[Dict, Dict, Any]],
    predict_fn: Callable[[Dict, Dict, Dict], np.ndarray],
    target_key: str = "y_close_reg",
    window_names: Optional[List[str]] = None,
    ic_method: str = "spearman", n_shuffles: int = 1000,
    min_samples: int = 10, seed: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """يقيس فرضية واحدة عبر مخرجات ``rolling_splits`` مباشرة — لكل نافذة
    ``(train, val, test)``: ``predict_fn(train, val, test)`` تُنتج تنبؤات
    بطول ``test`` (بعد تسطيحه إن لزم)، تُقارَن بـ``test[target_key]``.

    ``predict_fn`` قد تتجاهل ``train``/``val`` كلياً (فرضية يدوية بلا تدريب،
    كـ``momentum_predict_fn``)، أو تُدرِّب نموذجاً على ``train`` — العقد نفسه
    في الحالتين.
    """
    window_names = window_names or [f"نافذة {i+1}" for i in range(len(windows))]
    if len(window_names) != len(windows):
        raise ValueError("عدد الأسماء يجب أن يطابق عدد النوافذ.")

    results = []
    for name, (train, val, test) in zip(window_names, windows):
        test_flat = concat_splits(test)
        preds = predict_fn(train, val, test)
        actuals = extract_actuals(test_flat, target_key=target_key)
        if len(preds) != len(actuals):
            raise ValueError(
                f"[{name}] طول التنبؤات ({len(preds)}) لا يطابق طول الأهداف "
                f"({len(actuals)}) — راجع predict_fn.")
        results.append((name, np.asarray(preds, dtype="float64"), actuals))

    return evaluate_windows(results, ic_method=ic_method, n_shuffles=n_shuffles,
                            min_samples=min_samples, seed=seed, verbose=verbose)


## ٧) اختبارات ذاتية

كل دالة أعلاه مُختبَرة ببيانات تركيبية بإشارة معروفة الحجم مُحقَنة عمداً — لا اعتماد على بيانات حقيقية أو تشغيل دفتر التحضير. شغّلها بعد أي تعديل.

In [ ]:
# @title
# -*- coding: utf-8 -*-
import numpy as np
import pandas as pd

PASS, FAIL = [], []


def check(name, fn):
    try:
        fn()
        PASS.append(name)
        print(f"  ✅ {name}")
    except Exception as e:
        FAIL.append((name, f"{type(e).__name__}: {e}"))
        print(f"  ❌ {name}: {type(e).__name__}: {str(e)[:200]}")


# ══════════════════════════════════════════════════════════════════════════
# compute_ic
# ══════════════════════════════════════════════════════════════════════════

def t_ic_recovers_known_correlation():
    rng = np.random.default_rng(0)
    n = 5000
    true_signal = rng.normal(0, 1, n)
    actuals = 0.03 * true_signal + rng.normal(0, 1, n)   # IC حقيقي صغير ~0.03 (بمعيار سبيرمان تقريباً)
    preds = true_signal + rng.normal(0, 0.1, n)
    ic = compute_ic(preds, actuals, method="spearman")
    assert 0.01 < ic < 0.06, f"IC={ic} خارج المدى المتوقَّع"


def t_ic_near_zero_on_independent_random_data():
    rng = np.random.default_rng(1)
    n = 3000
    preds = rng.normal(0, 1, n)
    actuals = rng.normal(0, 1, n)
    ic = compute_ic(preds, actuals)
    se = 1.0 / np.sqrt(n)               # خطأ معياري تقريبي لارتباط سبيرمان
    assert abs(ic) < 4 * se, f"IC={ic} أكبر من المتوقَّع للضجيج المحض"


def t_ic_ignores_nan_not_zeros_them():
    p = np.array([1.0, 2.0, np.nan, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0])
    a = np.array([1.0, 2.0, 3.0, np.nan, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0])
    ic = compute_ic(p, a, min_samples=5)
    assert abs(ic - 1.0) < 1e-6, f"يجب أن يكون الارتباط شبه تام بعد استبعاد NaN، حصل {ic}"


def t_ic_raises_below_min_samples():
    p, a = np.array([1.0, 2.0, 3.0]), np.array([1.0, 2.0, 3.0])
    try:
        compute_ic(p, a, min_samples=10)
    except ValueError as e:
        assert "10" in str(e)
        return
    raise AssertionError("كان يجب أن يرفض عدداً أقل من الحدّ الأدنى")


def t_ic_spearman_robust_to_monotonic_nonlinear_transform():
    rng = np.random.default_rng(2)
    n = 2000
    p = rng.uniform(1, 10, n)
    a = np.exp(p) + rng.normal(0, 0.01, n)          # تحويل غير خطي لكن رتيب تماماً
    ic_s = compute_ic(p, a, method="spearman")
    ic_p = compute_ic(p, a, method="pearson")
    assert ic_s > 0.99, f"سبيرمان يجب أن يلتقط العلاقة الرتيبة تماماً، حصل {ic_s}"
    assert ic_p < ic_s, "بيرسون يجب أن يكون أضعف على علاقة غير خطية بهذا الحدّة"


def t_ic_zero_when_one_array_constant():
    p = np.full(20, 5.0)
    a = np.arange(20.0)
    assert compute_ic(p, a, min_samples=5) == 0.0


def t_ic_rejects_mismatched_shapes():
    try:
        compute_ic(np.zeros(5), np.zeros(6), min_samples=1)
    except ValueError:
        return
    raise AssertionError("أشكال مختلفة يجب أن تُرفَض")


def t_ic_rejects_unknown_method():
    try:
        compute_ic(np.zeros(20), np.zeros(20), method="kendall", min_samples=1)
    except ValueError as e:
        assert "kendall" in str(e)
        return
    raise AssertionError("طريقة غير معروفة يجب أن تُرفَض")


# ══════════════════════════════════════════════════════════════════════════
# decile_spread
# ══════════════════════════════════════════════════════════════════════════

def t_decile_detects_monotonic_relationship():
    rng = np.random.default_rng(3)
    n = 2000
    p = rng.normal(0, 1, n)
    a = p + rng.normal(0, 0.3, n)
    res = decile_spread(p, a, n_deciles=10, min_per_decile=20)
    assert res["monotonic"] is True
    assert res["spread"] > 0.5
    assert len(res["deciles"]) == 10
    assert res["deciles"]["n"].sum() == n


def t_decile_near_zero_spread_on_random_data():
    rng = np.random.default_rng(4)
    n = 3000
    p, a = rng.normal(0, 1, n), rng.normal(0, 1, n)
    res = decile_spread(p, a, min_per_decile=50)
    assert abs(res["spread"]) < 0.3, f"فرق العُشر {res['spread']} كبير جداً للضجيج المحض"


def t_decile_raises_on_insufficient_samples():
    p, a = np.arange(20.0), np.arange(20.0)
    try:
        decile_spread(p, a, n_deciles=10, min_per_decile=5)   # يحتاج 50، متوفر 20
    except ValueError:
        return
    raise AssertionError("عيّنات قليلة جداً لكل عُشر يجب أن تُرفَض")


def t_decile_t_stat_large_for_strong_signal_small_for_none():
    rng = np.random.default_rng(5)
    n = 1000
    p_strong = rng.normal(0, 1, n)
    a_strong = 2.0 * p_strong + rng.normal(0, 0.5, n)
    res_strong = decile_spread(p_strong, a_strong, min_per_decile=20)

    p_none, a_none = rng.normal(0, 1, n), rng.normal(0, 1, n)
    res_none = decile_spread(p_none, a_none, min_per_decile=20)

    assert abs(res_strong["top_minus_bottom_t_stat"]) > abs(res_none["top_minus_bottom_t_stat"])
    assert abs(res_strong["top_minus_bottom_t_stat"]) > 5


# ══════════════════════════════════════════════════════════════════════════
# permutation_baseline
# ══════════════════════════════════════════════════════════════════════════

def t_permutation_flags_real_signal_as_significant():
    rng = np.random.default_rng(6)
    n = 1500
    signal = rng.normal(0, 1, n)
    p = signal + rng.normal(0, 0.2, n)
    a = signal + rng.normal(0, 0.2, n)
    res = permutation_baseline(p, a, n_shuffles=300, seed=42)
    assert res["p_value"] < 0.01, f"p_value={res['p_value']} — إشارة قوية يجب أن تكون معنوية بوضوح"
    assert res["percentile"] > 95


def t_permutation_null_on_pure_random_data():
    rng = np.random.default_rng(7)
    n = 500
    p, a = rng.normal(0, 1, n), rng.normal(0, 1, n)
    res = permutation_baseline(p, a, n_shuffles=500, seed=1)
    assert res["p_value"] > 0.05, f"p_value={res['p_value']} — ضجيج محض لا يجب أن يبدو معنوياً"


def t_permutation_reproducible_with_seed():
    rng = np.random.default_rng(8)
    p, a = rng.normal(0, 1, 200), rng.normal(0, 1, 200)
    r1 = permutation_baseline(p, a, n_shuffles=100, seed=99)
    r2 = permutation_baseline(p, a, n_shuffles=100, seed=99)
    assert r1["null_mean"] == r2["null_mean"] and r1["p_value"] == r2["p_value"]


# ══════════════════════════════════════════════════════════════════════════
# evaluate_windows
# ══════════════════════════════════════════════════════════════════════════

def t_evaluate_windows_aggregates_consistent_signal():
    rng = np.random.default_rng(9)
    windows = []
    for i in range(5):
        n = 800
        signal = rng.normal(0, 1, n)
        p = signal + rng.normal(0, 1, n)
        a = 0.05 * signal + rng.normal(0, 1, n)
        windows.append((f"w{i}", p, a))
    res = evaluate_windows(windows, n_shuffles=200, seed=1, verbose=False)
    assert res["n_ok"] == 5
    assert res["mean_ic"] > 0
    assert res["consistent_sign"] is True


def t_evaluate_windows_skips_failed_window_not_whole_run():
    rng = np.random.default_rng(10)
    good = [(f"w{i}", rng.normal(0, 1, 500), rng.normal(0, 1, 500)) for i in range(3)]
    bad = [("w_bad", np.array([1.0, 2.0]), np.array([1.0, 2.0]))]   # عيّنات قليلة جداً
    res = evaluate_windows(good + bad, min_samples=10, n_shuffles=100, verbose=False)
    assert res["n_ok"] == 3
    assert (res["per_window"]["status"] == "skipped").sum() == 1


def t_evaluate_windows_no_signal_gives_low_frac_significant():
    rng = np.random.default_rng(11)
    windows = [(f"w{i}", rng.normal(0, 1, 600), rng.normal(0, 1, 600)) for i in range(5)]
    res = evaluate_windows(windows, n_shuffles=200, seed=2, verbose=False)
    assert res["frac_significant"] <= 0.4, f"ضجيج محض أعطى نوافذ معنوية كثيرة: {res['frac_significant']}"


# ══════════════════════════════════════════════════════════════════════════
# طبقة التكامل
# ══════════════════════════════════════════════════════════════════════════

def _fake_split(n=200, seed=0, feature_order=None, y_keys=("y_close_reg", "y_close_class")):
    """يحاكي الشكل الفعلي لقسم من _take (تحقّقتُه من المصدر مباشرة):
    الأهداف متداخلة تحت مفتاح فرعي واحد 'y' — لا مفاتيح y_head مباشرة على
    القسم نفسه. لا 'feature_order' (غير موجودة في أي قسم فعلياً، فقط في
    dataset الأصلي قبل التقطيع) — أي اختبار يحتاجها يمرّرها صراحةً.
    """
    rng = np.random.default_rng(seed)
    feature_order = feature_order or ["close", "RSI_14", "volume"]
    T, F = 32, len(feature_order)
    X = rng.normal(0, 1, (n, T, F)).astype("float32")
    y = {}
    for k in y_keys:
        y[k] = (np.sign(rng.normal(0, 1, n)) if k.endswith("_class")
                else rng.normal(0, 0.05, n)).astype("float32")
    return {"X_1D": X, "y": y, "base_params": np.zeros((n, 2), "float32"),
           "last_candles": np.zeros((n, 7), "float64")}


def t_concat_splits_passthrough_on_flat_dict():
    s = _fake_split()
    out = concat_splits(s)
    assert out is s


def t_concat_splits_nested_y_key_not_mistaken_for_asset_dict():
    """s['y'] قاموس أيضاً — إن اعتمد التمييز على isinstance(value, dict) فقط
    قد يُلتبَس بقاموس أصول؛ التمييز الفعلي عبر 'base_params' يتفادى هذا."""
    s = _fake_split(n=30, seed=99)
    assert isinstance(s["y"], dict)          # التبس محتمل لو اعتمدنا نوع القيمة فقط
    out = concat_splits(s)
    assert out is s


def t_concat_splits_merges_dict_of_assets():
    s1, s2 = _fake_split(n=50, seed=1), _fake_split(n=30, seed=2)
    merged = concat_splits({"AAAUSDT": s1, "BBBUSDT": s2})
    assert merged["X_1D"].shape[0] == 80
    assert merged["y"]["y_close_reg"].shape[0] == 80
    np.testing.assert_array_equal(merged["X_1D"][:50], s1["X_1D"])
    np.testing.assert_array_equal(merged["X_1D"][50:], s2["X_1D"])
    np.testing.assert_array_equal(merged["y"]["y_close_reg"][:50], s1["y"]["y_close_reg"])


def t_concat_splits_not_fooled_by_asset_names_starting_with_x():
    """يحاكي الفخّ الفعلي: اسم عملة مثل XRPUSDT قد يبدو كأنه مفتاح X_*
    لتخمين بالاسم — التمييز عبر 'base_params' لا يقع في هذا الفخّ."""
    s1, s2 = _fake_split(n=20, seed=20), _fake_split(n=15, seed=21)
    merged = concat_splits({"XRPUSDT": s1, "X_WEIRDUSDT": s2})
    assert merged["X_1D"].shape[0] == 35
    assert merged["y"]["y_close_reg"].shape[0] == 35


def t_concat_splits_raises_clear_error_when_asset_split_malformed():
    bad = {"AAAUSDT": {"some_other_key": np.zeros(5)}}      # بلا 'base_params'
    try:
        concat_splits(bad)
    except ValueError as e:
        assert "AAAUSDT" in str(e)
        return
    raise AssertionError("قسم أصل بلا 'base_params' يجب أن يُرفَض بوضوح")


def t_extract_actuals_reads_target_key():
    s = _fake_split(n=40, seed=3)
    a = extract_actuals(s, target_key="y_close_reg")
    np.testing.assert_array_equal(a, s["y"]["y_close_reg"].astype("float64"))


def t_extract_actuals_matches_with_or_without_y_prefix():
    """add_y_prefix قد يكون طُبِّق (prefix_y=True، الافتراضي) أو لا —
    يجب أن تعمل extract_actuals بالحالتين بلا تعديل من المستخدم."""
    s_prefixed = _fake_split(n=30, seed=30, y_keys=("y_close_reg",))
    a1 = extract_actuals(s_prefixed, target_key="close_reg")     # بلا بادئة
    np.testing.assert_array_equal(a1, s_prefixed["y"]["y_close_reg"].astype("float64"))

    s_unprefixed = _fake_split(n=30, seed=31, y_keys=("close_reg",))
    a2 = extract_actuals(s_unprefixed, target_key="y_close_reg")  # مع بادئة
    np.testing.assert_array_equal(a2, s_unprefixed["y"]["close_reg"].astype("float64"))


def t_extract_actuals_raises_on_missing_key():
    s = _fake_split(n=10, seed=4)
    try:
        extract_actuals(s, target_key="y_missing")
    except KeyError as e:
        assert "y_close_reg" in str(e)
        return
    raise AssertionError("مفتاح مفقود يجب أن يُرفَض بوضوح")


def t_extract_actuals_raises_clear_error_without_y_key():
    s = {"X_1D": np.zeros((5, 3, 2), "float32"), "base_params": np.zeros((5, 2), "float32"),
        "last_candles": np.zeros((5, 7))}     # بلا 'y' إطلاقاً
    try:
        extract_actuals(s)
    except ValueError as e:
        assert "'y'" in str(e)
        return
    raise AssertionError("قسم بلا مفتاح 'y' يجب أن يُرفَض بوضوح لا بخطأ غامض")


def t_extract_feature_last_diff_matches_manual():
    # feature_order غير موجودة على القسم فعلياً (مؤكَّد من الشكل الحقيقي) —
    # تُمرَّر صراحةً، كما في الاستخدام الفعلي دائماً.
    s = _fake_split(n=15, seed=5, feature_order=["a", "close", "b"])
    diff = extract_feature_last_diff(s, feature="close", tf="1D",
                                     feature_order=["a", "close", "b"])
    expected = s["X_1D"][:, -1, 1] - s["X_1D"][:, -2, 1]
    np.testing.assert_allclose(diff, expected, atol=1e-6)


def t_extract_feature_last_diff_requires_explicit_feature_order():
    """الحالة الواقعية: نوافذ rolling_splits لا تحمل feature_order إطلاقاً —
    يجب أن تُرفَض بوضوح بلا تمريرها، وتنجح عند تمريرها صراحةً."""
    s = _fake_split(n=12, seed=14, feature_order=["a", "close", "b"])
    assert "feature_order" not in s                  # الحالة الواقعية الافتراضية الآن
    try:
        extract_feature_last_diff(s, feature="close", tf="1D")
    except ValueError as e:
        assert "feature_order" in str(e)
    else:
        raise AssertionError("غياب feature_order يجب أن يُرفَض بوضوح")
    diff = extract_feature_last_diff(s, feature="close", tf="1D",
                                     feature_order=["a", "close", "b"])
    expected = s["X_1D"][:, -1, 1] - s["X_1D"][:, -2, 1]
    np.testing.assert_allclose(diff, expected, atol=1e-6)


def t_momentum_predict_fn_requires_explicit_feature_order():
    s = _fake_split(n=20, seed=15, feature_order=["close", "RSI_14", "volume"])
    preds = momentum_predict_fn(None, None, s, feature_order=["close", "RSI_14", "volume"])
    assert len(preds) == 20 and np.isfinite(preds).all()


def t_extract_feature_last_diff_raises_on_unknown_feature():
    s = _fake_split(n=10, seed=6)
    try:
        extract_feature_last_diff(s, feature="GHOST", feature_order=["close", "RSI_14", "volume"])
    except KeyError:
        return
    raise AssertionError("ميزة غير موجودة يجب أن تُرفَض")


def t_momentum_predict_fn_end_to_end_no_signal():
    s = _fake_split(n=500, seed=7)
    preds = momentum_predict_fn(None, None, s, feature_order=["close", "RSI_14", "volume"])
    assert len(preds) == 500 and np.isfinite(preds).all()


def t_evaluate_hypothesis_over_rolling_windows_detects_injected_signal():
    """محاكاة كاملة لبوّابة المرحلة ٠: زخم مُحقَن فعلياً في بيانات تركيبية
    تحاكي شكل مخرجات rolling_splits — يجب أن يُكتشَف بثبات عبر النوافذ."""
    rng = np.random.default_rng(8)
    windows = []
    for w in range(4):
        n = 600
        X = rng.normal(0, 1, (n, 32, 3)).astype("float32")
        momentum = X[:, -1, 0] - X[:, -2, 0]              # "آخر تغيّر" في close
        y = 0.04 * momentum + rng.normal(0, 1, n).astype("float32")   # استمرار ضعيف حقيقي
        test = {"X_1D": X, "y": {"y_close_reg": y}, "feature_order": ["close", "RSI_14", "volume"],
               "base_params": np.zeros((n, 2), "float32"), "last_candles": np.zeros((n, 7))}
        windows.append(({}, {}, test))
    res = evaluate_hypothesis_over_rolling_windows(
        windows, momentum_predict_fn, n_shuffles=200, seed=3, verbose=False)
    assert res["n_ok"] == 4
    assert res["mean_ic"] > 0.01
    assert res["consistent_sign"] is True


def t_evaluate_hypothesis_over_rolling_windows_no_signal_case():
    rng = np.random.default_rng(9)
    windows = []
    for w in range(4):
        n = 600
        X = rng.normal(0, 1, (n, 32, 3)).astype("float32")
        y = rng.normal(0, 1, n).astype("float32")          # عائد مستقلّ تماماً عن أي ميزة
        test = {"X_1D": X, "y": {"y_close_reg": y}, "feature_order": ["close", "RSI_14", "volume"],
               "base_params": np.zeros((n, 2), "float32"), "last_candles": np.zeros((n, 7))}
        windows.append(({}, {}, test))
    res = evaluate_hypothesis_over_rolling_windows(
        windows, momentum_predict_fn, n_shuffles=200, seed=4, verbose=False)
    assert res["frac_significant"] <= 0.5
    assert abs(res["mean_ic"]) < 0.1


def t_evaluate_hypothesis_handles_asset_separated_test():
    from functools import partial
    s1 = _fake_split(n=100, seed=11)
    s2 = _fake_split(n=80, seed=12)
    windows = [({}, {}, {"AAAUSDT": s1, "BBBUSDT": s2})]
    predict_fn = partial(momentum_predict_fn, feature_order=["close", "RSI_14", "volume"])
    res = evaluate_hypothesis_over_rolling_windows(
        windows, predict_fn, n_shuffles=50, min_samples=10, verbose=False)
    assert res["n_ok"] == 1


def t_evaluate_hypothesis_raises_on_length_mismatch():
    s = _fake_split(n=50, seed=13)
    bad_predict = lambda train, val, test: np.zeros(10)     # طول خاطئ عمداً
    try:
        evaluate_hypothesis_over_rolling_windows([({}, {}, s)], bad_predict, verbose=False)
    except ValueError as e:
        assert "طول" in str(e)
        return
    raise AssertionError("عدم تطابق الطول يجب أن يُرفَض بوضوح")


ALL_TESTS = [
    t_ic_recovers_known_correlation, t_ic_near_zero_on_independent_random_data,
    t_ic_ignores_nan_not_zeros_them, t_ic_raises_below_min_samples,
    t_ic_spearman_robust_to_monotonic_nonlinear_transform, t_ic_zero_when_one_array_constant,
    t_ic_rejects_mismatched_shapes, t_ic_rejects_unknown_method,
    t_decile_detects_monotonic_relationship, t_decile_near_zero_spread_on_random_data,
    t_decile_raises_on_insufficient_samples, t_decile_t_stat_large_for_strong_signal_small_for_none,
    t_permutation_flags_real_signal_as_significant, t_permutation_null_on_pure_random_data,
    t_permutation_reproducible_with_seed,
    t_evaluate_windows_aggregates_consistent_signal, t_evaluate_windows_skips_failed_window_not_whole_run,
    t_evaluate_windows_no_signal_gives_low_frac_significant,
    t_concat_splits_passthrough_on_flat_dict, t_concat_splits_merges_dict_of_assets,
    t_concat_splits_not_fooled_by_asset_names_starting_with_x,
    t_concat_splits_raises_clear_error_when_asset_split_malformed,
    t_concat_splits_nested_y_key_not_mistaken_for_asset_dict,
    t_extract_actuals_reads_target_key, t_extract_actuals_raises_on_missing_key,
    t_extract_actuals_matches_with_or_without_y_prefix,
    t_extract_actuals_raises_clear_error_without_y_key,
    t_extract_feature_last_diff_matches_manual, t_extract_feature_last_diff_raises_on_unknown_feature,
    t_extract_feature_last_diff_requires_explicit_feature_order,
    t_momentum_predict_fn_requires_explicit_feature_order,
    t_momentum_predict_fn_end_to_end_no_signal,
    t_evaluate_hypothesis_over_rolling_windows_detects_injected_signal,
    t_evaluate_hypothesis_over_rolling_windows_no_signal_case,
    t_evaluate_hypothesis_handles_asset_separated_test,
    t_evaluate_hypothesis_raises_on_length_mismatch,
]

PASS.clear(); FAIL.clear()
for t in ALL_TESTS:
    check(t.__name__, t)
print(f"\n{'✅ نجحت كل الاختبارات' if not FAIL else '❌ فشل بعضها'} ({len(PASS)}/{len(ALL_TESTS)})")
if FAIL:
    raise AssertionError([n for n, _ in FAIL])


  ✅ t_ic_recovers_known_correlation
  ✅ t_ic_near_zero_on_independent_random_data
  ✅ t_ic_ignores_nan_not_zeros_them
  ✅ t_ic_raises_below_min_samples
  ✅ t_ic_spearman_robust_to_monotonic_nonlinear_transform
  ✅ t_ic_zero_when_one_array_constant
  ✅ t_ic_rejects_mismatched_shapes
  ✅ t_ic_rejects_unknown_method
  ✅ t_decile_detects_monotonic_relationship
  ✅ t_decile_near_zero_spread_on_random_data
  ✅ t_decile_raises_on_insufficient_samples
  ✅ t_decile_t_stat_large_for_strong_signal_small_for_none
  ✅ t_permutation_flags_real_signal_as_significant
  ✅ t_permutation_null_on_pure_random_data
  ✅ t_permutation_reproducible_with_seed
  ✅ t_evaluate_windows_aggregates_consistent_signal
  ✅ t_evaluate_windows_skips_failed_window_not_whole_run
  ✅ t_evaluate_windows_no_signal_gives_low_frac_significant
  ✅ t_concat_splits_passthrough_on_flat_dict
  ✅ t_concat_splits_merges_dict_of_assets
  ✅ t_concat_splits_not_fooled_by_asset_names_starting_with_x
  ✅ t_concat_splits_raises_clear_error_w

In [ ]:
import gdown
import os,keras
import pandas as pd

def download_notebook_from_drive(
    file_id: str,
    notebook_name: str = 'dataprocess.ipynb',
    download_dir: str = '.',
    quiet: bool = False
) -> str:
    """
    تحميل notebook من Google Drive

    Args:
        file_id: معرّف الملف (يُستخرج من رابط المشاركة)
        notebook_name: اسم الملف المحلي
        download_dir: مجلد الحفظ
        quiet: إخفاء تفاصيل التحميل

    Returns:
        المسار الكامل للملف المحمّل
    """
    # ✅ تنظيف file_id من أي معاملات إضافية
    file_id = file_id.split('?')[0].strip()

    output_path = os.path.join(download_dir, notebook_name)
    url = f'https://drive.google.com/uc?id={file_id}'

    print(f"📥 جاري تحميل {notebook_name}...")

    try:
        gdown.download(url, output_path, quiet=quiet, fuzzy=True)
    except Exception as e:
        print(f"❌ خطأ في التحميل: {e}")
        print(f"💡 تأكد من:")
        print(f"   1. الملف مشارك بإعداد 'Anyone with the link'")
        print(f"   2. file_id صحيح: {file_id}")
        print(f"   3. الرابط الكامل: {url}")
        raise

    if not os.path.exists(output_path):
        raise FileNotFoundError(f"❌ فشل التحميل: {output_path}")

    file_size = os.path.getsize(output_path) / 1024
    print(f"✅ تم التحميل: {output_path} ({file_size:.1f} KB)")

    return output_path


file_id = '1zH1dYlcJclHVHsq1GkWcFxDaUIYbrRPg'  # بدون ?usp=sharing
path = download_notebook_from_drive(file_id)
# لتنفيذ كود من notebook آخر داخل notebook الحالي
%run dataprocess.ipynb

# save_data_to_drive(dataset)

dataset=load_preprocessed_data_from_drive("1g745bxEKHMcEDMtJ4B8cPMunuQ7wNKGH")

📥 جاري تحميل dataprocess.ipynb...


Downloading...
From: https://drive.google.com/uc?id=1zH1dYlcJclHVHsq1GkWcFxDaUIYbrRPg
To: /content/dataprocess.ipynb
100%|██████████| 402k/402k [00:00<00:00, 53.5MB/s]


✅ تم التحميل: ./dataprocess.ipynb (392.9 KB)
  ✅ t_exclude_matching
  ✅ t_exclude_edge_cases
  ❌ t_exclude_in_pipeline: Exception: Version mismatch: this is the 'cffi' package version 2.0.0, located in '/usr/local/lib/python3.13/dist-packages/cffi/api.py'.  When we import the top-level '_cf
  ❌ t_hour_features_daily: Exception: Version mismatch: this is the 'cffi' package version 2.0.0, located in '/usr/local/lib/python3.13/dist-packages/cffi/api.py'.  When we import the top-level '_cf
  ✅ t_hour_features_intraday
  ✅ t_split_crowded_timeline
  ✅ t_split_uniform_matches_requested
  ✅ t_split_raises_when_impossible
  ✅ t_split_explicit_dates_guarded
  ✅ t_fetch_single_page_unchanged
  ✅ t_fetch_paginates_and_merges
  ✅ t_fetch_exact_multiple_of_max
  ✅ t_fetch_history_shorter_than_limit
  ✅ t_fetch_retries_transient_page_error
  ✅ t_fetch_returns_none_on_persistent_page_failure
  ✅ t_fetch_rejects_nonpositive_limit
  ❌ t_exclude_features_names_and_patterns: Exception: Version mismatch: 

AssertionError: فشل 12 من 82 اختباراً: ['t_exclude_in_pipeline', 't_hour_features_daily', 't_exclude_features_names_and_patterns', 't_exclude_features_idempotent_and_guard', 't_exclude_features_reaches_dataset', 't_reg_target_matches_direct_return', 't_reg_target_uses_own_kind_not_last_close', 't_reg_target_clip_default_depends_on_mode', 't_build_dataset_live_decoupled_intervals_end_to_end', 't_market_context_reaches_dataset_via_build_dataset', 't_build_dataset_from_loader_checkpoint_resume_end_to_end', 't_build_dataset_checkpoint_invalidated_by_config_change']

AssertionError: فشل 12 من 82 اختباراً: ['t_exclude_in_pipeline', 't_hour_features_daily', 't_exclude_features_names_and_patterns', 't_exclude_features_idempotent_and_guard', 't_exclude_features_reaches_dataset', 't_reg_target_matches_direct_return', 't_reg_target_uses_own_kind_not_last_close', 't_reg_target_clip_default_depends_on_mode', 't_build_dataset_live_decoupled_intervals_end_to_end', 't_market_context_reaches_dataset_via_build_dataset', 't_build_dataset_from_loader_checkpoint_resume_end_to_end', 't_build_dataset_checkpoint_invalidated_by_config_change']

In [ ]:
dataset=load_preprocessed_data_from_drive("1g745bxEKHMcEDMtJ4B8cPMunuQ7wNKGH")

🔄 جاري التحميل من Google Drive...


Downloading...
From (original): https://drive.google.com/uc?id=1g745bxEKHMcEDMtJ4B8cPMunuQ7wNKGH
From (redirected): https://drive.google.com/uc?id=1g745bxEKHMcEDMtJ4B8cPMunuQ7wNKGH&confirm=t&uuid=f01ea3bd-08cd-4079-b00b-3004df2d7c0e
To: /content/preprocessing_output.pkl.gz
100%|██████████| 819M/819M [00:10<00:00, 77.3MB/s]


✅ تم التحميل! نوع: <class 'dict'>
🔑 المفاتيح: 21


## ٨) بوّابة المرحلة ٠ — التشغيل الفعلي على بياناتك

شغّل هذا **بعد** تشغيل `crypto_data_pipeline_v6.ipynb` في نفس الجلسة (أو بعد `%run crypto_data_pipeline_v6.ipynb`)، بحيث تكون `rolling_splits`, `build_dataset`, `CONFIG` متاحة.

هذا القسم مثال توضيحي (لن يعمل حرفياً بلا بيانات حقيقية مُحمَّلة) — عدّل `dataset`/الإعدادات حسب مشروعك الفعلي.

In [ ]:
# مثال — يفترض أن dataset (مخرجات build_dataset) وCONFIG متاحان من تشغيل دفتر التحضير
from functools import partial

windows = rolling_splits(dataset, test_span="30D", val_span="15D",
                         initial_train_span="200D", max_windows=30, config=CONFIG)

# ✅ feature_order غير محفوظة داخل نوافذ rolling_splits (قِيس فعلياً على بيانات
# حقيقية) — تُمرَّر صراحةً من dataset الأصلي عبر partial، لا الاعتماد على
# وجودها في كل قسم train/val/test.
predict_fn = partial(momentum_predict_fn, feature_order=dataset["feature_order"])

report = evaluate_hypothesis_over_rolling_windows(
    windows, predict_fn, target_key="y_close_reg",
    n_shuffles=1000, seed=42)

report["per_window"]   # جدول تفصيلي لكل نافذة

# إن أعطى mean_ic > 0 صغيراً لكن consistent_sign=True عبر أغلب النوافذ:
# المحور موثوق — انتقل لاختبار فرضيات المرحلة ١ (funding rate، open interest...).
# إن لم يُعطِ: المشكلة في القياس نفسه (تحقّق من target_key، محاذاة الفريم،
# أو جرّب momentum بفترة أطول/أقصر) قبل اختبار أي فرضية أخرى.


🔁 نافذة 1/30: train≤2020-05-30 (1,500) | val≤2020-07-17 (345) | test≤2020-09-18 (846)
🔁 نافذة 2/30: train≤2020-06-29 (2,190) | val≤2020-08-16 (360) | test≤2020-10-18 (951)
🔁 نافذة 3/30: train≤2020-07-29 (2,880) | val≤2020-09-15 (436) | test≤2020-11-17 (1,101)
🔁 نافذة 4/30: train≤2020-08-28 (3,619) | val≤2020-10-15 (491) | test≤2020-12-17 (1,322)
🔁 نافذة 5/30: train≤2020-09-27 (4,499) | val≤2020-11-14 (568) | test≤2021-01-16 (1,490)
🔁 نافذة 6/30: train≤2020-10-27 (5,488) | val≤2020-12-14 (672) | test≤2021-02-15 (1,573)
🔁 نافذة 7/30: train≤2020-11-26 (6,664) | val≤2021-01-13 (750) | test≤2021-03-17 (1,663)
🔁 نافذة 8/30: train≤2020-12-26 (8,043) | val≤2021-02-12 (798) | test≤2021-04-16 (1,736)
🔁 نافذة 9/30: train≤2021-01-25 (9,549) | val≤2021-03-14 (840) | test≤2021-05-16 (1,810)
🔁 نافذة 10/30: train≤2021-02-24 (11,156) | val≤2021-04-13 (884) | test≤2021-06-15 (2,050)
🔁 نافذة 11/30: train≤2021-03-26 (12,833) | val≤2021-05-13 (915) | test≤2021-07-15 (2,132)
🔁 نافذة 12/30: train≤2021-04-25 

,window,status,ic,spread,monotonic,p_value,percentile
0,نافذة 1,ok,-0.290746,-0.028182,False,0.000,0.0
1,نافذة 2,ok,0.026592,-0.003820,False,0.425,79.1
2,نافذة 3,ok,-0.084153,-0.005970,False,0.007,0.4
3,نافذة 4,ok,-0.102203,-0.015976,False,0.000,0.0
4,نافذة 5,ok,-0.020439,-0.007911,False,0.422,21.5
5,نافذة 6,ok,-0.081024,-0.021007,False,0.000,0.0
6,نافذة 7,ok,-0.135428,-0.016676,False,0.000,0.0
7,نافذة 8,ok,-0.068493,-0.018452,False,0.005,0.2
8,نافذة 9,ok,-0.127898,-0.014369,False,0.000,0.0
9,نافذة 10,ok,-0.180442,-0.159693,False,0.000,0.0


## ٩) سجلّ التجارب (Experiment Registry)

المكوّن الثالث من "خطة بناء النظام" في خطة المشروع — لم يكن مبنياً بعد. كل
فرضية تُختبَر عبر محور هذا الدفتر تحصل على سجلّ واحد دائم: معرّف، صياغة
بسطر واحد، المصدر (أيّ من مسارات الاكتشاف الأربعة)، الحالة، ونتائج المحور
**كاملة** — لا رقم ملخّص وحده، بل جدول كل نافذة أيضاً، لأن رقماً واحداً
("IC=-0.08") لا يكفي للحكم لاحقاً على نافذة شاذّة أو نمط زمني فيها.

القاعدة الحاكمة (من الخطة): لا حالة وسطى دائمة. "قيد الاختبار" عابرة فقط
حتى ينتهي التشغيل الفعلي؛ بعدها الفرضية إمّا **مقبولة** أو **مرفوضة**،
موثَّقة في الحالتين.

In [ ]:
# -*- coding: utf-8 -*-
"""سجلّ التجارب — محفوظ كملف JSON (لا قاعدة بيانات) بنفس روح checkpoint
العمل الحالي في دفتر التحضير: بصيغة قابلة للقراءة والتحكّم اليدوي، وتُكتب
على القرص بعد كل تسجيل فوراً (لا في الذاكرة فقط) فتنجو من فقدان جلسة Colab.
"""
from __future__ import annotations
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional
import json


#: المسارات الأربعة من قسم "منهجية الاكتشاف" في خطة المشروع — قيمة `source`
#: موحّدة، لا نصاً حراً، ليبقى الحقل قابلاً للتصفية لاحقاً.
DISCOVERY_TRACKS = (
    "hypothesis_driven",   # أ) مبنيّ على الفرضية
    "data_driven",         # ب) مبنيّ على البيانات
    "literature_mining",   # ج) مبنيّ على الأدبيات
    "genetic_search",      # د) البحث الموجَّه
)

#: لا حالة وسطى دائمة (راجع خطة المشروع، قسم "لماذا مشروع منفصل").
REGISTRY_STATUSES = ("قيد الاختبار", "مقبولة", "مرفوضة")


def _default_registry_path(config: Optional[dict] = None) -> Path:
    """داخل مجلد المشروع على Drive إن كان مركَّباً (نفس drive_mount_point في
    دفتر التحضير)، وإلا محلياً بجانب الدفتر — يعمل بلا Drive للاختبارات."""
    config = config or {}
    base = config.get("drive_mount_point")
    d = (Path(base) / "MyDrive" / config.get("project_name", "crypto_model")
         / "experiment_registry") if base and Path(base).exists() else Path("experiment_registry")
    d.mkdir(parents=True, exist_ok=True)
    return d / "registry.json"


def _summarize_report(report: Dict[str, Any]) -> Dict[str, Any]:
    """يستخلص من مخرجات evaluate_windows/evaluate_hypothesis_over_rolling_windows
    كل ما يلزم لحكم مستقل لاحقاً — per_window كاملة، لا الملخّص وحده."""
    per_window = report.get("per_window")
    if per_window is not None and hasattr(per_window, "to_dict"):
        per_window = per_window.to_dict(orient="records")
    return {"mean_ic": report.get("mean_ic"), "std_ic": report.get("std_ic"),
            "frac_significant": report.get("frac_significant"),
            "consistent_sign": report.get("consistent_sign"),
            "n_ok": report.get("n_ok"), "per_window": per_window}


def _load_entries(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        return []
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def _save_entries(path: Path, entries: List[Dict[str, Any]]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(entries, f, ensure_ascii=False, indent=2)


def register_hypothesis(hyp_id: str, hypothesis: str, source: str, status: str,
                        report: Optional[Dict[str, Any]] = None, notes: str = "",
                        registry_path: Optional[Path] = None,
                        config: Optional[dict] = None) -> Dict[str, Any]:
    """يُضيف سجلّاً جديداً، أو يستبدل سجلّاً بنفس ``hyp_id`` بالكامل إن وُجد
    (لا دمج جزئي — يمنع حقولاً قديمة متبقية من نتيجة سابقة).

    ``source``/``status`` تُرفَض بخطأ صريح إن خرجتا عن :data:`DISCOVERY_TRACKS`/
    :data:`REGISTRY_STATUSES` — قيمة حرّة هنا تكسر أي تصفية لاحقة على السجلّ.
    """
    if source not in DISCOVERY_TRACKS:
        raise ValueError(f"source يجب أن يكون أحد {DISCOVERY_TRACKS} (وصل {source!r}).")
    if status not in REGISTRY_STATUSES:
        raise ValueError(f"status يجب أن يكون أحد {REGISTRY_STATUSES} (وصل {status!r}).")

    path = registry_path or _default_registry_path(config)
    entries = _load_entries(path)
    entry = {"id": hyp_id, "hypothesis": hypothesis, "source": source, "status": status,
             "notes": notes, "axis_results": _summarize_report(report) if report is not None else None,
             "registered_at": datetime.now(timezone.utc).isoformat()}
    entries = [e for e in entries if e["id"] != hyp_id] + [entry]
    _save_entries(path, entries)
    print(f"\u2705 \u0633\u064f\u062c\u0651\u0644\u062a '{hyp_id}' \u0628\u062d\u0627\u0644\u0629 '{status}' \u0641\u064a {path}")
    return entry


def list_registry(registry_path: Optional[Path] = None, config: Optional[dict] = None):
    """جدول موجز — عمود واحد لكل حقل أساسي، بلا per_window (طويلة، عرضها
    هنا سيغرق البقية). راجعها عبر ``get_hypothesis(id)['axis_results']['per_window']``."""
    import pandas as pd
    path = registry_path or _default_registry_path(config)
    entries = _load_entries(path)
    cols = ["id", "hypothesis", "source", "status", "mean_ic", "frac_significant", "consistent_sign"]
    if not entries:
        print(f"\u0627\u0644\u0633\u062c\u0644\u0651 \u0641\u0627\u0631\u063a \u0628\u0639\u062f ({path}).")
        return pd.DataFrame(columns=cols)
    rows = [{"id": e["id"], "hypothesis": e["hypothesis"], "source": e["source"], "status": e["status"],
             "mean_ic": (e.get("axis_results") or {}).get("mean_ic"),
             "frac_significant": (e.get("axis_results") or {}).get("frac_significant"),
             "consistent_sign": (e.get("axis_results") or {}).get("consistent_sign")} for e in entries]
    return pd.DataFrame(rows)[cols]


def get_hypothesis(hyp_id: str, registry_path: Optional[Path] = None,
                   config: Optional[dict] = None) -> Dict[str, Any]:
    path = registry_path or _default_registry_path(config)
    for e in _load_entries(path):
        if e["id"] == hyp_id:
            return e
    raise KeyError(f"\u0644\u0627 \u0633\u062c\u0644\u0651 \u0628\u0645\u0639\u0631\u0651\u0641 '{hyp_id}' \u0641\u064a {path}.")


### اختبارات ذاتية — سجلّ التجارب

In [ ]:
# @title
def run_registry_selftests() -> None:
    """اختبارات سجلّ التجارب — تعمل بلا Drive، على مسار مؤقّت."""
    import tempfile, shutil

    tmp = Path(tempfile.mkdtemp())
    reg = tmp / "registry.json"
    tests = 0
    failed = []

    def check(name, cond):
        nonlocal tests
        tests += 1
        if not cond:
            failed.append(name)

    try:
        # 1) تسجيل بسيط ثم قراءته
        register_hypothesis("H_test", "فرضية تجريبية", "hypothesis_driven",
                            "قيد الاختبار", registry_path=reg)
        e = get_hypothesis("H_test", registry_path=reg)
        check("تسجيل بسيط", e["hypothesis"] == "فرضية تجريبية" and e["status"] == "قيد الاختبار")

        # 2) استبدال كامل عند نفس المعرّف (لا دمج جزئي)
        register_hypothesis("H_test", "فرضية تجريبية", "hypothesis_driven",
                            "مقبولة", notes="بعد التحديث", registry_path=reg)
        e2 = get_hypothesis("H_test", registry_path=reg)
        check("استبدال كامل عند تكرار المعرّف",
              e2["status"] == "مقبولة" and e2["notes"] == "بعد التحديث")
        check("لا تكرار للسجلّ", len(_load_entries(reg)) == 1)

        # 3) source/status غير صالحين يُرفضان بخطأ صريح
        try:
            register_hypothesis("H_bad", "x", "not_a_track", "مقبولة", registry_path=reg)
            check("رفض source غير صالح", False)
        except ValueError:
            check("رفض source غير صالح", True)

        try:
            register_hypothesis("H_bad", "x", "hypothesis_driven", "not_a_status", registry_path=reg)
            check("رفض status غير صالح", False)
        except ValueError:
            check("رفض status غير صالح", True)

        # 4) report كامل (تقليد شكل evaluate_windows) يُختزَل ويُحفَظ بصورة قابلة لـJSON
        import pandas as pd
        fake_report = {"mean_ic": -0.05, "std_ic": 0.02, "frac_significant": 0.6,
                       "consistent_sign": False, "n_ok": 3,
                       "per_window": pd.DataFrame([{"window": "نافذة 1", "ic": -0.05}])}
        register_hypothesis("H_report", "فرضية بنتائج", "literature_mining",
                            "مقبولة", report=fake_report, registry_path=reg)
        e3 = get_hypothesis("H_report", registry_path=reg)
        check("per_window يُحفَظ كقائمة قواميس لا DataFrame",
              isinstance(e3["axis_results"]["per_window"], list))
        check("mean_ic محفوظ بصحّة", e3["axis_results"]["mean_ic"] == -0.05)

        # 5) list_registry يرجع جدولاً بعدد الصفوف الصحيح
        table = list_registry(registry_path=reg)
        check("list_registry بعدد الصفوف الصحيح", len(table) == 2)

        # 6) معرّف غير موجود يرفع KeyError، لا يرجع None بصمت
        try:
            get_hypothesis("لا_يوجد", registry_path=reg)
            check("get_hypothesis يرفع KeyError لمعرّف غائب", False)
        except KeyError:
            check("get_hypothesis يرفع KeyError لمعرّف غائب", True)

    finally:
        shutil.rmtree(tmp, ignore_errors=True)

    if failed:
        print(f"\u274c {len(failed)}/{tests} \u0641\u0634\u0644\u062a: {failed}")
    else:
        print(f"\u2705 \u0643\u0644 \u0627\u0644\u0627\u062e\u062a\u0628\u0627\u0631\u0627\u062a \u0646\u062c\u062d\u062a ({tests}/{tests}) \u2014 \u0633\u062c\u0644\u0651 \u0627\u0644\u062a\u062c\u0627\u0631\u0628.")


run_registry_selftests()


✅ سُجّلت 'H_test' بحالة 'قيد الاختبار' في /tmp/tmp6sb03edl/registry.json
✅ سُجّلت 'H_test' بحالة 'مقبولة' في /tmp/tmp6sb03edl/registry.json
✅ سُجّلت 'H_report' بحالة 'مقبولة' في /tmp/tmp6sb03edl/registry.json
✅ كل الاختبارات نجحت (9/9) — سجلّ التجارب.
